# Aula 01 - Introdução a Agentes de IA

Bem-vindo à primeira aula do curso **Agentes de IA para Iniciantes**!

Um **agente de IA** é um programa que utiliza um grande modelo de linguagem (LLM) como seu motor de raciocínio e pode tomar *ações* no mundo real — chamar APIs, consultar bases de dados ou executar código — para alcançar um objetivo em nome de um utilizador.

Neste caderno, irá construir o seu primeiro agente: um **Agente de Viagens** que recomenda destinos de férias. Ao longo do caminho, irá aprender a:

1. Ligar ao Azure AI Foundry Agent Service usando o **Microsoft Agent Framework**.
2. Dar ao agente uma **ferramenta** — uma função Python simples que pode chamar.
3. Executar o agente e inspecionar a sua resposta.
4. Transmitir a resposta do agente token a token.


## Configuração

Antes de executar este notebook, certifique-se de que tem:

1. **Um projeto Azure AI Foundry** com um modelo de chat implantado (por exemplo, `gpt-4o-mini`).
2. **Sessão iniciada com o Azure CLI** — execute `az login` no seu terminal.
3. **Definidas as variáveis de ambiente necessárias:**
   - `AZURE_AI_PROJECT_ENDPOINT` — o endpoint do seu projeto Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — o nome do seu modelo implantado.

A célula abaixo instala os pacotes Python que precisa.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Criar o Seu Primeiro Agente

Um agente precisa de duas coisas:

- **Instruções** que lhe dizem *quem é* e *como se deve comportar* (um prompt de sistema).
- **Ferramentas** — funções Python decoradas com `@tool` que o agente pode chamar para obter informações ou executar ações.

A seguir, definimos uma ferramenta simples que devolve uma lista de destinos de férias populares. O agente usará esta ferramenta quando um utilizador pedir recomendações de viagens.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Respostas em Streaming

Para uma experiência mais interativa pode **transmitir** a resposta do agente. Em vez de esperar pela resposta completa, o agente devolve fragmentos de texto à medida que são gerados. Isto é especialmente útil em interfaces de chat onde se pretende mostrar a saída em tempo real.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Resumo

Nesta lição aprendeu a:

- **Criar um fornecedor** que se liga ao Azure AI Foundry Agent Service através do `AzureAIProjectAgentProvider`.
- **Definir uma ferramenta** usando o decorador `@tool` para que o agente possa chamar as suas funções Python.
- **Executar o agente** com uma mensagem do utilizador e imprimir a sua resposta.
- **Transmissão de respostas** para saída em tempo real.

Na próxima lição iremos explorar frameworks agentícios com mais profundidade e aprender como dar aos agentes ferramentas mais poderosas e capacidades de raciocínio em vários passos.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento foi traduzido utilizando o serviço de tradução automatizada [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos para garantir a precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original no seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se a tradução humana profissional. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
